# Loss functions and optimizers

_PyTorch (a complete course)_

**The two pieces that turn a model into learning: a loss to score it, an optimizer to fix it.**

Learning is a feedback loop. The loss answers "how wrong are we?" as a single number. Calling
        loss.backward() fills every parameter's .grad with the slope of that number
        with respect to that parameter. The optimizer then takes one downhill step.

       You attach the optimizer to the model's parameters once. From then on it remembers where they live and,
        for adaptive methods like Adam, keeps running statistics (a moving average of past gradients) to size each
        step automatically.

---

This notebook is a practice scaffold for the **Loss functions and optimizers** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)  # reproducible

# --- tiny synthetic 3-class problem: 60 points, 4 features ---
N, D_in, C = 60, 4, 3
X = torch.randn(N, D_in)
# integer class labels 0/1/2 -- NOT one-hot. CrossEntropyLoss wants indices.
y = torch.randint(0, C, (N,))            # shape (N,), dtype long

# --- model: outputs raw LOGITS of shape (N, C). NO softmax at the end! ---
model = nn.Sequential(
    nn.Linear(D_in, 16),
    nn.ReLU(),
    nn.Linear(16, C),                    # last layer -> raw logits, that's it
)

# loss + optimizer. Adam is given the model's parameters and a learning rate.
criterion = nn.CrossEntropyLoss()        # applies log-softmax INTERNALLY
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

for step in range(20):
    logits = model(X)                    # (N, C) raw logits
    loss = criterion(logits, y)          # pass logits + integer targets

    optimizer.zero_grad()                # 1. clear old grads (else they accumulate)
    loss.backward()                      # 2. backprop: fill every .grad
    optimizer.step()                     # 3. one downhill step

    if step % 4 == 0:
        acc = (logits.argmax(1) == y).float().mean().item()
        print(f"step {step:2d}  loss {loss.item():.4f}  train_acc {acc:.2f}")

# --- the GOTCHA, shown ---
# WRONG: softmax before CrossEntropyLoss -> double softmax, training degrades.
#   bad = nn.CrossEntropyLoss()(torch.softmax(logits, dim=1), y)
# WRONG: one-hot targets -> CrossEntropyLoss wants class indices, not one-hot.
#   y_onehot = torch.nn.functional.one_hot(y, C).float()  # do NOT pass this
# RIGHT: raw logits + integer indices, exactly as above.

print("done. note: the model has NO softmax layer -- CrossEntropyLoss adds it.")

## Visualize the data & results

_On the same tiny noisy problem, how do SGD and Adam compare step by step?_

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
N, d = 200, 8
# ill-conditioned features: columns scaled 1..20 so one direction is far steeper
X = rng.standard_normal((N, d)) * np.array([1, 1, 3, 3, 8, 8, 20, 20.0])
w_true = rng.standard_normal(d)
y = X @ w_true + 0.1 * rng.standard_normal(N)   # noisy linear target

def full_loss(w):
    r = X @ w - y
    return 0.5 * np.mean(r * r)                  # MSE over ALL data (what we plot)

def batch_grad(w, idx):
    Xb, yb = X[idx], y[idx]
    return Xb.T @ (Xb @ w - yb) / len(idx)       # mini-batch gradient (noisy)

steps, bs, w0 = 50, 16, np.zeros(d)

def run(opt, lr):
    w = w0.copy(); m = np.zeros(d); v = np.zeros(d); hist = []
    rs = np.random.default_rng(42)               # same batch stream for both
    for t in range(1, steps + 1):
        hist.append(full_loss(w))
        g = batch_grad(w, rs.integers(0, N, bs))
        if opt == "sgd":
            w = w - lr * g                        # plain SGD step
        else:                                     # Adam
            b1, b2, eps = 0.9, 0.999, 1e-8
            m = b1 * m + (1 - b1) * g
            v = b2 * v + (1 - b2) * g * g
            mh = m / (1 - b1**t); vh = v / (1 - b2**t)
            w = w - lr * mh / (np.sqrt(vh) + eps)
    return hist

sgd  = run("sgd",  0.0025)
adam = run("adam", 0.15)
print("final loss -- SGD:", round(sgd[-1], 3), " Adam:", round(adam[-1], 3))
# -> final loss -- SGD: 2.281  Adam: 1.008

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Your image classifier ends with nn.Softmax(dim=1) and you train it with
            nn.CrossEntropyLoss. Accuracy is stuck. What's wrong and how do you fix it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall what CrossEntropyLoss does to its input. — _It applies log-softmax internally, so it expects raw logits._
- Notice the model already applied softmax. — _You are now softmaxing twice — the gradients are tiny and learning stalls._

**Answer:** Remove the Softmax layer so the model outputs raw logits. CrossEntropyLoss
                 handles the softmax itself. (For inference, apply softmax separately if you need probabilities.)

</details>

**Problem 2.** You write the loop as loss.backward(); optimizer.step() with no
            zero_grad(). The loss behaves erratically. Why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall PyTorch's gradient behavior. — _.grad accumulates across backward calls; it is not reset automatically._
- Trace step 2. — _Its gradient is added on top of step 1's leftover gradient — the update direction is corrupted._

**Answer:** Call optimizer.zero_grad() at the top of every step, before
                 loss.backward(). Then each step uses only its own gradient.

</details>

**Problem 3.** Pick the loss: (a) predicting house price (a number), (b) predicting one of 10 digit classes,
            (c) tagging an image with any of 5 non-exclusive labels.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Match task type to loss. — _Regression &rarr; squared error; single-label &rarr; cross-entropy; independent labels &rarr; per-label binary._

**Answer:** (a) nn.MSELoss. (b) nn.CrossEntropyLoss on logits with integer targets.
                 (c) nn.BCEWithLogitsLoss on logits with multi-hot float targets.

</details>